<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp5_1_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# EXPERIMENT 5.1-A
# R=5000 | dense N-grid | exact + interpolated test N | tail-aware training
# SPARSE-LU ONLY — NO MATRIX INVERSE
#
# R0 = beta/gamma
# Specific-case PMF/tail figures display R0.
# =====================================================================================

from __future__ import annotations
import copy, hashlib, json, math, pickle, random, time
from dataclasses import dataclass, asdict
from functools import lru_cache
from pathlib import Path
from typing import Tuple

import numpy as np
from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc

import torch
import torch.nn as nn
import matplotlib.pyplot as plt


# =====================================================================================
# 1. CONFIGURATION
# =====================================================================================

@dataclass
class Config:
    seed:int=20260820

    beta_range:Tuple[float,float]=(.30,1.50)
    gamma_range:Tuple[float,float]=(.20,1.00)
    omega_range:Tuple[float,float]=(.02,.50)
    initial_fraction_range:Tuple[float,float]=(.02,.20)
    i0_one_fraction:float=.25

    train_N:Tuple[int,...]=(40,60,80,100,120,140,160,180,200,220,240,260,280,300,320,340,360,380,400)
    interp_N:Tuple[int,...]=(60,90,120,150,180,210,240,270,300,330,360,390)

    n_train:int=10000
    n_val:int=500
    n_test_exact:int=500
    n_test_interp:int=450

    width:int=128
    depth:int=3
    batch_size:int=64
    epochs:int=500

    lr:float=1e-3
    weight_decay:float=1e-6

    lambda_rho:float=1.0
    lambda_tau:float=.02

    patience:int=50
    min_delta:float=1e-6
    grad_clip:float=5.

    prob_tol:float=1e-10
    var_rel_tol:float=1e-8
    kl_eps:float=1e-12
    refine_steps:int=3

    teacher_version:str="R5000_tailaware_sparseLU_R0_v11"
    output_dir:str="results_section_5_1A_R5000_tailaware_R0"
    dpi:int=300


cfg=Config()

EXACT_N=cfg.train_N
INTERP_N=cfg.interp_N
ALL_TEST_N=tuple(sorted(EXACT_N+INTERP_N))


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


seed_all(cfg.seed)

device=torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

out=Path(cfg.output_dir)
out.mkdir(parents=True,exist_ok=True)

tag=hashlib.sha1(
    json.dumps(asdict(cfg),sort_keys=True).encode()
).hexdigest()[:12]

N_scale=max(cfg.train_N)

print("="*110)
print("EXPERIMENT 5.1-A — R=5000, TAIL-AWARE")
print("="*110)
print("Device:",device)
print("Train N:",cfg.train_N)
print("Exact test N:",EXACT_N)
print("Interpolated test N:",INTERP_N)
print("Total distinct test N:",len(ALL_TEST_N))
print("R0 = beta/gamma")
print("No matrix inverse is formed.")


# =====================================================================================
# 2. CTMC TOPOLOGY
# =====================================================================================

@lru_cache(maxsize=None)
def topology(N):

    states=[
        (s,i)
        for i in range(1,N+1)
        for s in range(N-i+1)
    ]

    idx={x:j for j,x in enumerate(states)}
    M=len(states)

    ir=[]; ic=[]; ib=[]
    rr=[]; rc=[]; rb=[]
    wr=[]; wc=[]; wb=[]

    db=np.zeros(M)
    dg=np.zeros(M)
    dw=np.zeros(M)
    qb=np.zeros(M)

    for row,(s,i) in enumerate(states):

        r=N-s-i

        if s:
            ir.append(row)
            ic.append(idx[(s-1,i+1)])
            ib.append(s*i/N)
            db[row]=s*i/N

        dg[row]=i

        if i==1:
            qb[row]=i
        else:
            rr.append(row)
            rc.append(idx[(s,i-1)])
            rb.append(i)

        if r:
            wr.append(row)
            wc.append(idx[(s+1,i)])
            wb.append(r)
            dw[row]=r

    return (
        idx,M,
        np.asarray(ir),np.asarray(ic),np.asarray(ib,float),
        np.asarray(rr),np.asarray(rc),np.asarray(rb,float),
        np.asarray(wr),np.asarray(wc),np.asarray(wb,float),
        db,dg,dw,qb
    )


def matrices(beta,gamma,omega,N):

    (
        idx,M,
        ir,ic,ib,
        rr,rc,rb,
        wr,wc,wb,
        db,dg,dw,qb
    )=topology(N)

    rows=np.concatenate([
        ir,rr,wr,np.arange(M)
    ])

    cols=np.concatenate([
        ic,rc,wc,np.arange(M)
    ])

    vals=np.concatenate([
        beta*ib,
        gamma*rb,
        omega*wb,
        -(beta*db+gamma*dg+omega*dw)
    ])

    T=sparse.coo_matrix(
        (vals,(rows,cols)),
        shape=(M,M),
        dtype=np.float64
    ).tocsc()

    D1=sparse.coo_matrix(
        (beta*ib,(ir,ic)),
        shape=(M,M),
        dtype=np.float64
    ).tocsc()

    q=gamma*qb

    return T,(T-D1).tocsc(),D1,q,idx


# =====================================================================================
# 3. SPARSE LU
# =====================================================================================

def factor(A,ordering="COLAMD"):
    return splu(
        A.tocsc(),
        permc_spec=ordering
    )


def solve_lu(A,lu,b,transpose=False):

    b=np.asarray(b,dtype=np.float64)
    mode="T" if transpose else "N"

    x=lu.solve(
        b,
        trans=mode
    )

    for _ in range(cfg.refine_steps):

        r=b-(A.T@x if transpose else A@x)

        if not np.all(np.isfinite(r)):
            break

        rel=(
            np.linalg.norm(r,np.inf)
            /
            max(np.linalg.norm(b,np.inf),1.)
        )

        if rel<1e-11:
            break

        x+=lu.solve(
            r,
            trans=mode
        )

    return np.asarray(x,dtype=np.float64)


def moment_factor(A,ordering):

    d=np.abs(A.diagonal())

    scale=1./np.maximum(
        d,
        np.finfo(float).tiny
    )

    As=(
        sparse.diags(scale)
        @
        A
    ).tocsc()

    return (
        splu(
            As,
            permc_spec=ordering
        ),
        scale
    )


def moment_solve(A,lu,scale,b):

    b=np.asarray(b,dtype=np.float64)

    x=lu.solve(
        scale*b
    )

    for _ in range(cfg.refine_steps):

        r=b-A@x

        if not np.all(np.isfinite(r)):
            break

        rel=(
            np.linalg.norm(r,np.inf)
            /
            max(np.linalg.norm(b,np.inf),1.)
        )

        if rel<1e-11:
            break

        x+=lu.solve(
            scale*r
        )

    return np.asarray(x,dtype=np.float64)


# =====================================================================================
# 4. EXTINCTION MOMENTS
# =====================================================================================

def variance_system(T,q,A,lu,scale,m1):

    C=T.tocoo()
    keep=C.row!=C.col

    rr=C.row[keep]
    cc=C.col[keep]
    rates=C.data[keep]

    src=np.bincount(
        rr,
        weights=rates*(m1[cc]-m1[rr])**2,
        minlength=T.shape[0]
    ).astype(np.float64)

    src+=q*m1*m1

    if (
        not np.all(np.isfinite(src))
        or src.min()<-1e-8
    ):
        raise ArithmeticError(
            "Invalid variance-system RHS"
        )

    return moment_solve(
        A,lu,scale,
        np.maximum(src,0.)
    )


def extinction_moments(T,q,initial):

    A=(-T).tocsc()
    one=np.ones(A.shape[0])

    errors=[]

    for ordering in (
        "COLAMD",
        "MMD_AT_PLUS_A"
    ):

        try:

            lu,scale=moment_factor(
                A,ordering
            )

            # (-T)m1 = 1
            m1=moment_solve(
                A,lu,scale,one
            )

            mean=float(
                m1[initial]
            )

            if (
                not np.all(np.isfinite(m1))
                or mean<=0
            ):
                raise ArithmeticError(
                    f"invalid E(tau)={mean}"
                )

            # (-T)m2 = 2m1
            m2=moment_solve(
                A,lu,scale,2*m1
            )

            second=float(
                m2[initial]
            )

            if (
                not np.all(np.isfinite(m2))
                or second<=0
            ):
                raise ArithmeticError(
                    f"invalid E(tau^2)={second}"
                )

            raw=(
                np.longdouble(second)
                -
                np.longdouble(mean)**2
            )

            tol=(
                cfg.var_rel_tol
                *
                max(
                    second,
                    mean*mean,
                    1.
                )
            )

            if (
                np.isfinite(raw)
                and raw>=-tol
            ):

                var=max(
                    float(raw),
                    0.
                )

                method="subtraction"

            else:

                vv=variance_system(
                    T,q,A,lu,scale,m1
                )

                var=float(
                    vv[initial]
                )

                method="linear-system"

            if (
                not np.isfinite(var)
                or var<0
            ):
                raise ArithmeticError(
                    f"invalid Var(tau)={var}"
                )

            return (
                mean,
                second,
                var,
                True,
                f"{ordering}; {method}"
            )

        except Exception as e:

            errors.append(
                f"{ordering}: {e}"
            )

    return (
        np.nan,
        np.nan,
        np.nan,
        False,
        " | ".join(errors)
    )


# =====================================================================================
# 5. EXACT TEACHER
# =====================================================================================

def exact_targets(beta,gamma,omega,N,i0):

    T,D0,D1,q,idx=matrices(
        beta,gamma,omega,N
    )

    M=T.shape[0]
    initial=idx[(N-i0,i0)]

    # Infection-count distribution
    A0=(-D0).tocsc()
    lu0=factor(A0)

    alpha=np.zeros(M)
    alpha[initial]=1.

    b=solve_lu(
        A0,lu0,q
    )

    v=alpha.copy()
    p=np.zeros(N+2)

    for k in range(N+1):

        p[k]=v@b

        y=solve_lu(
            A0,lu0,v,
            transpose=True
        )

        v=np.asarray(
            D1.T@y
        ).ravel()

    p[-1]=v.sum()

    p[
        np.abs(p)<cfg.prob_tol
    ]=0.

    if (
        not np.all(np.isfinite(p))
        or
        p.min()<-cfg.prob_tol
    ):
        raise RuntimeError(
            f"Invalid PMF: N={N}, i0={i0}"
        )

    p=np.maximum(
        p,0.
    )

    mass=p.sum()

    if (
        not np.isfinite(mass)
        or
        abs(mass-1)>1e-5
    ):
        raise RuntimeError(
            f"Invalid PMF mass={mass}"
        )

    p/=mass

    mean,second,var,ok,info=extinction_moments(
        T,q,initial
    )

    return (
        p,mean,second,var,ok,info
    )


# =====================================================================================
# 6. DESIGN
# =====================================================================================

@dataclass
class Record:
    beta:float
    gamma:float
    omega:float

    N:int
    i0:int

    p:np.ndarray

    mean_tau:float
    second_tau:float
    var_tau:float

    tau_valid:bool
    tau_info:str


def stretch(u,bounds):

    a,b=bounds

    return a+(b-a)*u


def design(n,Ns,seed):

    U=qmc.LatinHypercube(
        d=4,
        seed=seed
    ).random(n)

    beta=stretch(
        U[:,0],
        cfg.beta_range
    )

    gamma=stretch(
        U[:,1],
        cfg.gamma_range
    )

    omega=stretch(
        U[:,2],
        cfg.omega_range
    )

    frac=stretch(
        U[:,3],
        cfg.initial_fraction_range
    )

    Nv=np.tile(
        np.asarray(Ns),
        math.ceil(n/len(Ns))
    )[:n]

    rng=np.random.default_rng(
        seed+991
    )

    rng.shuffle(Nv)

    i0=np.asarray([
        int(
            np.clip(
                round(frac[j]*Nv[j]),
                2,
                Nv[j]
            )
        )
        for j in range(n)
    ])

    for N in Ns:

        ix=np.where(
            Nv==N
        )[0]

        k=max(
            1,
            int(
                round(
                    cfg.i0_one_fraction
                    *
                    len(ix)
                )
            )
        )

        i0[
            rng.choice(
                ix,k,
                replace=False
            )
        ]=1

    return [
        (
            float(beta[j]),
            float(gamma[j]),
            float(omega[j]),
            int(Nv[j]),
            int(i0[j])
        )
        for j in range(n)
    ]


def make_dataset(configs,name):

    ans=[]
    unresolved=0
    t0=time.perf_counter()

    for j,(b,g,w,N,i0) in enumerate(
        configs,1
    ):

        try:

            p,m,m2,v,ok,info=exact_targets(
                b,g,w,N,i0
            )

        except Exception as e:

            raise RuntimeError(
                f"\nEXACT PMF TEACHER FAILURE\n"
                f"{name}, record {j}\n"
                f"N={N}, i0={i0}, "
                f"beta={b:.7g}, gamma={g:.7g}, omega={w:.7g}\n"
                f"{e}"
            ) from e

        unresolved+=int(not ok)

        ans.append(
            Record(
                b,g,w,N,i0,p,
                m,m2,v,
                ok,info
            )
        )

        if j%25==0 or j==len(configs):

            print(
                f"[{name:12s}] "
                f"{j:5d}/{len(configs):5d} | "
                f"N={N:3d}, i0={i0:3d} | "
                f"tau unresolved={unresolved:3d} | "
                f"{time.perf_counter()-t0:.1f}s"
            )

    return ans


def load_or_make(name,configs):

    path=out/f"{name}_{cfg.teacher_version}_{tag}.pkl"

    if path.exists():

        print(
            "Loading",path
        )

        with open(path,"rb") as f:
            return pickle.load(f)

    x=make_dataset(
        configs,name
    )

    with open(path,"wb") as f:
        pickle.dump(x,f)

    return x


train=load_or_make(
    "train",
    design(
        cfg.n_train,
        cfg.train_N,
        cfg.seed+1
    )
)

valid=load_or_make(
    "validation",
    design(
        cfg.n_val,
        cfg.train_N,
        cfg.seed+2
    )
)

test_exact=load_or_make(
    "test_exact",
    design(
        cfg.n_test_exact,
        EXACT_N,
        cfg.seed+3
    )
)

test_interp=load_or_make(
    "test_interp",
    design(
        cfg.n_test_interp,
        INTERP_N,
        cfg.seed+4
    )
)


# =====================================================================================
# 7. AUDIT
# =====================================================================================

def audit(records,name):

    print("\n"+name)

    for N in sorted(
        {r.N for r in records}
    ):

        z=[
            r for r in records
            if r.N==N
        ]

        print(
            f"N={N:3d} | "
            f"n={len(z):3d} | "
            f"i0=1={sum(r.i0==1 for r in z):3d} | "
            f"valid tau={sum(r.tau_valid for r in z):3d}/{len(z)}"
        )


print("\n"+"="*110)
print("EXACT-TARGET AUDIT")
print("="*110)

audit(train,"TRAIN")
audit(valid,"VALIDATION")
audit(test_exact,"EXACT TEST")
audit(test_interp,"INTERPOLATION TEST")


# =====================================================================================
# 8. NETWORKS
# =====================================================================================

def mlp(din,dout):

    L=[]
    d=din

    for _ in range(cfg.depth):

        L += [
            nn.Linear(
                d,cfg.width
            ),
            nn.SiLU()
        ]

        d=cfg.width

    L.append(
        nn.Linear(
            d,dout
        )
    )

    return nn.Sequential(*L)


class HazardNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.net=mlp(6,1)

    def forward(self,x):
        return torch.sigmoid(
            self.net(x).squeeze(-1)
        )


class TauNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.net=mlp(5,2)

    def forward(self,x):
        return torch.nn.functional.softplus(
            self.net(x)
        )


def xtau(r):

    return torch.tensor(
        [
            r.beta,
            r.gamma,
            r.omega,
            r.N/N_scale,
            r.i0/r.N
        ],
        dtype=torch.float32,
        device=device
    )


def xhaz(r):

    c=torch.arange(
        r.N+1,
        dtype=torch.float32,
        device=device
    )

    n=r.N+1

    return torch.column_stack([
        torch.full((n,),r.beta,device=device),
        torch.full((n,),r.gamma,device=device),
        torch.full((n,),r.omega,device=device),
        torch.full((n,),r.N/N_scale,device=device),
        torch.full((n,),r.i0/r.N,device=device),
        c/r.N
    ])


def reconstruct(h):

    before=torch.cat([
        torch.ones(
            1,
            device=h.device
        ),
        torch.cumprod(
            1-h[:-1],
            0
        )
    ])

    return torch.cat([
        before*h,
        torch.prod(1-h).reshape(1)
    ])


def tail_torch(p):

    return torch.flip(
        torch.cumsum(
            torch.flip(
                p[1:],
                dims=[0]
            ),
            dim=0
        ),
        dims=[0]
    )


def phat_tensor(net,r):
    return reconstruct(
        net(xhaz(r))
    )


def tau_target(r):

    raw=np.asarray(
        [
            r.mean_tau,
            r.var_tau
        ],
        dtype=np.float64
    )

    z=np.log1p(raw)

    return torch.tensor(
        z,
        dtype=torch.float32,
        device=device
    )


# =====================================================================================
# 9. TAIL-AWARE LOSS
# =====================================================================================

def batch_loss(records,ix,hnet,tnet):

    Lp=[]
    Lrho=[]
    Lt=[]

    for j in ix:

        r=records[int(j)]

        p=torch.tensor(
            r.p,
            dtype=torch.float32,
            device=device
        )

        ph=phat_tensor(
            hnet,r
        )

        # PMF loss
        Lp.append(
            torch.sum(
                (ph-p)**2
            )
        )

        # Tail-aware loss
        rho=tail_torch(p)
        rhoh=tail_torch(ph)

        Lrho.append(
            torch.mean(
                (rhoh-rho)**2
            )
        )

        if r.tau_valid:

            z=tau_target(r)

            zh=tnet(
                xtau(r).unsqueeze(0)
            ).squeeze(0)

            Lt.append(
                torch.sum(
                    (zh-z)**2
                    /
                    (1+z*z)
                )
            )

    lp=torch.stack(Lp).mean()
    lrho=torch.stack(Lrho).mean()

    lt=(
        torch.stack(Lt).mean()
        if Lt
        else
        torch.zeros(
            (),
            device=device
        )
    )

    return (
        lp
        +
        cfg.lambda_rho*lrho
        +
        cfg.lambda_tau*lt,
        lp,lrho,lt
    )


# =====================================================================================
# 10. TRAIN
# =====================================================================================

@torch.no_grad()
def val_loss(hnet,tnet):

    hnet.eval()
    tnet.eval()

    L,_,_,_=batch_loss(
        valid,
        np.arange(len(valid)),
        hnet,
        tnet
    )

    return L.item()


def train_model():

    seed_all(
        cfg.seed+10000
    )

    hnet=HazardNet().to(device)
    tnet=TauNet().to(device)

    pars=(
        list(hnet.parameters())
        +
        list(tnet.parameters())
    )

    opt=torch.optim.AdamW(
        pars,
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=.5,
        patience=15
    )

    rng=np.random.default_rng(
        cfg.seed+7001
    )

    best=np.inf
    bh=bt=None
    wait=0

    hist={
        "joint":[],
        "pmf":[],
        "tail":[],
        "tau":[],
        "val":[]
    }

    print("\n"+"#"*110)
    print("TRAINING R=5000 TAIL-AWARE EMULATOR")
    print("#"*110)

    t0=time.perf_counter()

    for epoch in range(
        1,
        cfg.epochs+1
    ):

        hnet.train()
        tnet.train()

        perm=rng.permutation(
            len(train)
        )

        J=[];P=[];RHO=[];T=[]

        for s in range(
            0,
            len(perm),
            cfg.batch_size
        ):

            ix=perm[
                s:s+cfg.batch_size
            ]

            opt.zero_grad()

            L,Lp,Lrho,Lt=batch_loss(
                train,ix,hnet,tnet
            )

            if not torch.isfinite(L):

                raise RuntimeError(
                    f"Non-finite loss at epoch={epoch}"
                )

            L.backward()

            torch.nn.utils.clip_grad_norm_(
                pars,
                cfg.grad_clip
            )

            opt.step()

            J.append(L.detach().item())
            P.append(Lp.detach().item())
            RHO.append(Lrho.detach().item())
            T.append(Lt.detach().item())

        j=np.mean(J)
        p=np.mean(P)
        rh=np.mean(RHO)
        ta=np.mean(T)

        v=val_loss(
            hnet,tnet
        )

        sch.step(v)

        hist["joint"].append(j)
        hist["pmf"].append(p)
        hist["tail"].append(rh)
        hist["tau"].append(ta)
        hist["val"].append(v)

        if (
            bh is None
            or
            v<best-cfg.min_delta
        ):

            best=v

            bh=copy.deepcopy(
                hnet.state_dict()
            )

            bt=copy.deepcopy(
                tnet.state_dict()
            )

            wait=0

        else:

            wait+=1

        if (
            epoch==1
            or
            epoch%20==0
        ):

            print(
                f"epoch={epoch:4d} | "
                f"joint={j:.4e} | "
                f"pmf={p:.4e} | "
                f"tail={rh:.4e} | "
                f"tau={ta:.4e} | "
                f"val={v:.4e}"
            )

        if wait>=cfg.patience:
            print("Early stopping:",epoch)
            break

    hnet.load_state_dict(bh)
    tnet.load_state_dict(bt)

    return (
        hnet,
        tnet,
        hist,
        time.perf_counter()-t0,
        best
    )


hnet,tnet,history,training_time,best_val=train_model()


# =====================================================================================
# 11. EVALUATION
# =====================================================================================

def tail_np(p):
    return np.flip(
        np.cumsum(
            np.flip(p[1:])
        )
    )


def entropy(p):

    z=p[p>0]

    return float(
        -np.sum(
            z*np.log(z)
        )
    )


def bimodality(p):

    z=p[:-1]

    peaks=[
        j for j in range(len(z))
        if
        z[j]>=(z[j-1] if j else -np.inf)
        and
        z[j]>=(z[j+1] if j<len(z)-1 else -np.inf)
        and
        z[j]>=.03*z.max()
    ]

    if len(peaks)<2:
        return 0.

    a,b=sorted(
        peaks,
        key=lambda j:z[j],
        reverse=True
    )[:2]

    return float(
        min(z[a],z[b])
        /
        max(z[a],z[b])
        *
        abs(a-b)
        /
        max(len(z)-1,1)
    )


@torch.no_grad()
def predict(r):

    hnet.eval()
    tnet.eval()

    p=phat_tensor(
        hnet,r
    ).cpu().numpy()

    z=(
        tnet(
            xtau(r).unsqueeze(0)
        )
        .squeeze(0)
        .cpu()
        .numpy()
        .astype(np.float64)
    )

    mom=np.expm1(
        np.clip(
            z,
            0.,
            700.
        )
    )

    return (
        p,
        float(mom[0]),
        float(mom[1])
    )


def evaluate(records,split):

    ans=[]

    for j,r in enumerate(records):

        p,m,v=predict(r)

        rho=tail_np(r.p)
        rhoh=tail_np(p)

        positive=r.p>0

        ps=np.clip(
            p,
            cfg.kl_eps,
            1.
        )

        # Basic reproduction number
        R0=r.beta/r.gamma

        row={
            "index":j,
            "split":split,

            "N":r.N,
            "i0":r.i0,
            "i0_fraction":r.i0/r.N,

            "beta":r.beta,
            "gamma":r.gamma,
            "omega":r.omega,

            "R0":R0,

            "tau_valid":r.tau_valid,

            "E2":
                float(
                    np.linalg.norm(
                        p-r.p
                    )
                ),

            "E_rho":
                float(
                    np.max(
                        np.abs(
                            rhoh-rho
                        )
                    )
                ),

            "E_overflow":
                float(
                    abs(
                        p[-1]-r.p[-1]
                    )
                ),

            "KL":
                float(
                    np.sum(
                        r.p[positive]
                        *
                        np.log(
                            r.p[positive]
                            /
                            ps[positive]
                        )
                    )
                ),

            "exact_overflow":
                float(r.p[-1]),

            "entropy":
                entropy(r.p),

            "bimodality":
                bimodality(r.p),

            "exact_p":r.p,
            "pred_p":p,

            "exact_tail":rho,
            "pred_tail":rhoh
        }

        if r.tau_valid:

            row["mean_tau_relative_error"]=float(
                abs(m-r.mean_tau)
                /
                r.mean_tau
            )

            row["var_tau_relative_error"]=float(
                abs(v-r.var_tau)
                /
                max(r.var_tau,1e-300)
            )

        else:

            row["mean_tau_relative_error"]=np.nan
            row["var_tau_relative_error"]=np.nan

        ans.append(row)

    return ans


exact_results=evaluate(
    test_exact,
    "exact"
)

interp_results=evaluate(
    test_interp,
    "interpolation"
)

all_results=(
    exact_results
    +
    interp_results
)


# =====================================================================================
# 12. SUMMARIES
# =====================================================================================

METRICS=[
    "E2",
    "E_rho",
    "E_overflow",
    "KL",
    "mean_tau_relative_error",
    "var_tau_relative_error"
]


def finite(data,key):

    z=np.asarray([
        r[key]
        for r in data
    ],float)

    return z[
        np.isfinite(z)
    ]


def print_summary(data,title):

    print("\n"+"="*115)
    print(title)
    print("="*115)

    for key in METRICS:

        z=finite(
            data,key
        )

        print(
            f"{key:30s} | "
            f"median={np.median(z):.4e} | "
            f"IQR=({np.quantile(z,.25):.4e},{np.quantile(z,.75):.4e}) | "
            f"p95={np.quantile(z,.95):.4e} | "
            f"max={np.max(z):.4e} | "
            f"n={len(z)}"
        )


print_summary(
    exact_results,
    "TEST — EXACT N"
)

print_summary(
    interp_results,
    "TEST — INTERPOLATED N"
)

print_summary(
    all_results,
    "TEST — ALL N"
)


# =====================================================================================
# 13. FIGURE 1 — AGGREGATE ERROR BY N
# =====================================================================================

def byN(data,key):

    Ns=sorted({
        r["N"]
        for r in data
    })

    med=[]; q1=[]; q3=[]

    for N in Ns:

        z=np.asarray([
            r[key]
            for r in data
            if
            r["N"]==N
            and
            np.isfinite(r[key])
        ])

        med.append(
            np.median(z)
        )

        q1.append(
            np.quantile(z,.25)
        )

        q3.append(
            np.quantile(z,.75)
        )

    return (
        np.asarray(Ns),
        np.asarray(med),
        np.asarray(q1),
        np.asarray(q3)
    )


plt.rcParams.update({
    "font.size":10.5,
    "axes.spines.top":False,
    "axes.spines.right":False
})


fig,axs=plt.subplots(
    2,2,
    figsize=(13,9)
)

spec=[
    ("E2",r"Median $E_2$","(A) Distributional error"),
    ("E_rho",r"Median $E_\rho$","(B) Tail-risk error"),
    ("mean_tau_relative_error","Median relative error",r"(C) $E(\tau)$"),
    ("var_tau_relative_error","Median relative error",r"(D) $\mathrm{Var}(\tau)$")
]


for ax,(key,ylab,title) in zip(
    axs.flat,
    spec
):

    for data,label,col,mk in [
        (
            exact_results,
            "Exact-grid $N$",
            "#555555","o"
        ),
        (
            interp_results,
            "Interpolated $N$",
            "#0072B2","D"
        )
    ]:

        N,m,l,u=byN(
            data,key
        )

        ax.plot(
            N,
            np.maximum(m,1e-12),
            color=col,
            marker=mk,
            lw=1.8,
            label=label
        )

        ax.fill_between(
            N,
            np.maximum(l,1e-12),
            np.maximum(u,1e-12),
            color=col,
            alpha=.12
        )

    ax.set_yscale("log")
    ax.set_xlabel("Population size $N$")
    ax.set_ylabel(ylab)
    ax.set_title(title)
    ax.grid(alpha=.15)
    ax.legend(frameon=False)


fig.suptitle(
    "Predictive Accuracy Across Exact and Interpolated Population Sizes",
    fontsize=15
)

plt.tight_layout()

plt.savefig(
    out/"figure_5_1A_accuracy_vs_N.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 14. SPECIFIC CASE SELECTION
# =====================================================================================

median_rho=np.median([
    r["E_rho"]
    for r in interp_results
])

interp_i1=[
    r for r in interp_results
    if r["i0"]==1
]


cases=[
    (
        "Median interpolation",
        min(
            interp_results,
            key=lambda r:
                abs(
                    r["E_rho"]
                    -
                    median_rho
                )
        )
    ),

    (
        "Largest interpolation error",
        max(
            interp_results,
            key=lambda r:
                r["E_rho"]
        )
    ),

    (
        r"Interpolation, $i_0=1$",
        min(
            interp_i1,
            key=lambda r:
                abs(
                    r["E_rho"]
                    -
                    median_rho
                )
        )
    ),

    (
        "Highest-entropy interpolation",
        max(
            interp_results,
            key=lambda r:
                r["entropy"]
        )
    )
]


# =====================================================================================
# 15. FIGURE 2 — PMF EXAMPLES + R0
# =====================================================================================

fig,axs=plt.subplots(
    2,2,
    figsize=(14,9)
)


for ax,(label,r) in zip(
    axs.flat,
    cases
):

    c=np.arange(
        r["N"]+1
    )

    ax.bar(
        c,
        r["exact_p"][:-1],
        color=".82",
        width=.85,
        label="Exact Markovian"
    )

    ax.plot(
        c,
        r["pred_p"][:-1],
        color="#D55E00",
        lw=1.8,
        label="Neural emulator"
    )

    ax.set_title(
        f"{label}\n"
        f"$N={r['N']}$, "
        f"$i_0={r['i0']}$, "
        rf"$R_0={r['R0']:.2f}$, "
        rf"$E_2={r['E2']:.3f}$, "
        rf"$E_\rho={r['E_rho']:.3f}$"
    )

    ax.set_xlabel(
        "Infection count $c$"
    )

    ax.set_ylabel(
        "Probability mass"
    )

    ax.legend(
        frameon=False,
        fontsize=8
    )


plt.tight_layout()

plt.savefig(
    out/"figure_5_1A_pmf_gallery.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 16. FIGURE 3 — TAIL EXAMPLES + R0
# =====================================================================================

fig,axs=plt.subplots(
    2,2,
    figsize=(14,9)
)


for ax,(label,r) in zip(
    axs.flat,
    cases
):

    c=np.arange(
        r["N"]+1
    )

    ax.plot(
        c,
        r["exact_tail"],
        color="black",
        lw=2,
        label="Exact Markovian"
    )

    ax.plot(
        c,
        r["pred_tail"],
        "--",
        color="#0072B2",
        lw=1.8,
        label="Neural emulator"
    )

    ax.set_ylim(
        -.01,1.01
    )

    ax.set_title(
        f"{label}\n"
        f"$N={r['N']}$, "
        f"$i_0={r['i0']}$, "
        rf"$R_0={r['R0']:.2f}$, "
        rf"$E_\rho={r['E_rho']:.3f}$"
    )

    ax.set_xlabel(
        "Threshold $c$"
    )

    ax.set_ylabel(
        r"$P(C>c)$"
    )

    ax.legend(
        frameon=False,
        fontsize=8
    )


plt.tight_layout()

plt.savefig(
    out/"figure_5_1A_tail_gallery.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 17. GLOBAL CALIBRATION
# =====================================================================================

fig,axs=plt.subplots(
    1,3,
    figsize=(16,4.8)
)


def calibration(ax,x,y,title):

    x=np.clip(
        np.asarray(x),
        1e-10,None
    )

    y=np.clip(
        np.asarray(y),
        1e-10,None
    )

    lo=min(
        x.min(),y.min()
    )

    hi=max(
        x.max(),y.max()
    )

    ax.scatter(
        x,y,
        s=7,
        alpha=.15,
        color="#0072B2"
    )

    ax.plot(
        [lo,hi],
        [lo,hi],
        "k--",
        lw=1.2
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel("Exact")
    ax.set_ylabel("Neural")

    ax.set_title(title)


calibration(
    axs[0],
    np.concatenate([
        r["exact_p"]
        for r in interp_results
    ]),
    np.concatenate([
        r["pred_p"]
        for r in interp_results
    ]),
    "(A) Probability mass"
)

calibration(
    axs[1],
    np.concatenate([
        r["exact_tail"]
        for r in interp_results
    ]),
    np.concatenate([
        r["pred_tail"]
        for r in interp_results
    ]),
    "(B) Tail probability"
)

calibration(
    axs[2],
    [
        r["exact_overflow"]
        for r in interp_results
    ],
    [
        r["pred_p"][-1]
        for r in interp_results
    ],
    "(C) Overflow probability"
)


plt.tight_layout()

plt.savefig(
    out/"figure_5_1A_calibration.png",
    dpi=cfg.dpi,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 18. SAVE
# =====================================================================================

torch.save(
    {
        "hazard":
            hnet.state_dict(),

        "tau":
            tnet.state_dict(),

        "config":
            asdict(cfg),

        "training_time":
            training_time,

        "best_validation_loss":
            best_val
    },

    out/"section_5_1A_model.pt"
)


with open(
    out/"section_5_1A_results.pkl",
    "wb"
) as f:

    pickle.dump(
        {
            "config":
                asdict(cfg),

            "exact":
                exact_results,

            "interpolation":
                interp_results,

            "all_test":
                all_results,

            "history":
                history,

            "training_time":
                training_time,

            "best_validation_loss":
                best_val
        },
        f
    )


print("\n"+"="*110)
print("EXPERIMENT 5.1-A COMPLETE")
print("="*110)

print("Training R:",cfg.n_train)
print("Train N:",cfg.train_N)
print("Exact test N:",EXACT_N)
print("Interpolated N:",INTERP_N)
print("Tail-loss weight:",cfg.lambda_rho)
print("R0 = beta/gamma")

print(
    "Valid tau train:",
    sum(r.tau_valid for r in train),
    "/",
    len(train)
)

print(
    "Results:",
    out.resolve()
)

print("="*110)

EXPERIMENT 5.1-A — R=5000, TAIL-AWARE
Device: cpu
Train N: (40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400)
Exact test N: (40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400)
Interpolated test N: (60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360, 390)
Total distinct test N: 31
R0 = beta/gamma
No matrix inverse is formed.
[train       ]    25/ 5000 | N=100, i0=  1 | tau unresolved=  2 | 54.1s
[train       ]    50/ 5000 | N=180, i0=  1 | tau unresolved=  3 | 97.7s
[train       ]    75/ 5000 | N=120, i0= 18 | tau unresolved=  7 | 154.7s
[train       ]   100/ 5000 | N=380, i0= 11 | tau unresolved= 10 | 234.2s
[train       ]   125/ 5000 | N=120, i0= 10 | tau unresolved= 12 | 282.0s
[train       ]   150/ 5000 | N=400, i0= 73 | tau unresolved= 15 | 329.1s
[train       ]   175/ 5000 | N= 40, i0=  1 | tau unresolved= 17 | 380.9s
[train       ]   200/ 5000 | N=380, i0= 37 | tau unresolved= 17 | 438.2s
[train  

In [ ]:
# =====================================================================================
# FAILURE ANALYSIS — WORST 5% AND 10%
#
# Primary failure metric: E_rho
#
# Includes R0 = beta/gamma:
#   - in tables
#   - in regime diagnostics
#   - in worst-case PMF titles
#   - in worst-case tail titles
# =====================================================================================

from pathlib import Path
import pickle, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


RESULT_DIR=Path(
    "results_section_5_1A_R5000_tailaware_R0"
)

RESULT_FILE=(
    RESULT_DIR
    /
    "section_5_1A_results.pkl"
)

FAIL_METRIC="E_rho"
FRACTIONS=(.05,.10)


with open(
    RESULT_FILE,
    "rb"
) as f:

    D=pickle.load(f)


ALL=D["all_test"]
INTERP=D["interpolation"]


# =====================================================================================
# 1. DATA FRAMES
# =====================================================================================

def to_df(results):

    keep=[
        "index",
        "split",

        "N",
        "i0",
        "i0_fraction",

        "beta",
        "gamma",
        "omega",
        "R0",

        "E2",
        "E_rho",
        "E_overflow",
        "KL",

        "mean_tau_relative_error",
        "var_tau_relative_error",

        "exact_overflow",
        "entropy",
        "bimodality",
        "tau_valid"
    ]

    return pd.DataFrame([
        {
            k:r.get(k,np.nan)
            for k in keep
        }
        for r in results
    ])


df_all=to_df(ALL)
df_interp=to_df(INTERP)


# =====================================================================================
# 2. WORST FRACTIONS
# =====================================================================================

def worst_fraction(
    df,
    fraction,
    metric=FAIL_METRIC
):

    n=max(
        1,
        int(
            math.ceil(
                fraction*len(df)
            )
        )
    )

    return (
        df
        .sort_values(
            metric,
            ascending=False
        )
        .head(n)
        .copy()
    )


W={}


for scope,df in [
    ("all",df_all),
    ("interpolation",df_interp)
]:

    print("\n"+"="*115)
    print(scope.upper())
    print("="*115)

    for frac in FRACTIONS:

        w=worst_fraction(
            df,
            frac
        )

        W[(scope,frac)]=w

        print(
            f"\nWorst {int(frac*100)}% by {FAIL_METRIC}: "
            f"{len(w)}/{len(df)}"
        )

        print(
            f"E_rho: "
            f"median={w.E_rho.median():.4f}, "
            f"min={w.E_rho.min():.4f}, "
            f"max={w.E_rho.max():.4f}"
        )

        print(
            f"R0: "
            f"median={w.R0.median():.3f}, "
            f"IQR=({w.R0.quantile(.25):.3f},"
            f"{w.R0.quantile(.75):.3f}), "
            f"range=({w.R0.min():.3f},"
            f"{w.R0.max():.3f})"
        )

        print(
            "N counts:",
            w["N"]
            .value_counts()
            .sort_index()
            .to_dict()
        )

        print(
            f"i0=1 fraction: "
            f"{(w.i0==1).mean():.3f}"
        )

        print(
            f"median beta={w.beta.median():.3f}, "
            f"gamma={w.gamma.median():.3f}, "
            f"omega={w.omega.median():.3f}"
        )

        print(
            f"median overflow="
            f"{w.exact_overflow.median():.4f}, "
            f"entropy="
            f"{w.entropy.median():.3f}, "
            f"bimodality="
            f"{w.bimodality.median():.3f}"
        )

        w.to_csv(
            RESULT_DIR
            /
            f"worst_{int(frac*100)}pct_{scope}.csv",
            index=False
        )


# =====================================================================================
# 3. ALL TEST VS WORST 10%
# =====================================================================================

w10=W[("all",.10)]

compare_cols=[
    "N",
    "i0_fraction",

    "beta",
    "gamma",
    "omega",
    "R0",

    "exact_overflow",
    "entropy",
    "bimodality",

    "E2",
    "E_rho",
    "KL"
]


comparison=pd.DataFrame({

    "all_test_median":
        df_all[
            compare_cols
        ].median(),

    "worst10_median":
        w10[
            compare_cols
        ].median(),

    "worst10_Q25":
        w10[
            compare_cols
        ].quantile(.25),

    "worst10_Q75":
        w10[
            compare_cols
        ].quantile(.75)

})


print("\n"+"="*115)
print("ALL TEST VERSUS WORST 10%")
print("="*115)

display(comparison)


comparison.to_csv(
    RESULT_DIR
    /
    "failure_regime_summary.csv"
)


# =====================================================================================
# 4. FAILURE DIAGNOSTICS
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


# ------------------------------------------------------------------
# A. Distribution of E_rho
# ------------------------------------------------------------------

axs[0,0].hist(
    df_all.E_rho,
    bins=35,
    color=".72",
    edgecolor="white"
)

q90=df_all.E_rho.quantile(.90)
q95=df_all.E_rho.quantile(.95)

axs[0,0].axvline(
    q90,
    color="#E69F00",
    ls="--",
    label="90th percentile"
)

axs[0,0].axvline(
    q95,
    color="#D55E00",
    ls="--",
    label="95th percentile"
)

axs[0,0].set_xlabel(
    r"$E_\rho$"
)

axs[0,0].set_ylabel(
    "Number of configurations"
)

axs[0,0].set_title(
    "(A) Tail-error distribution"
)

axs[0,0].legend(
    frameon=False
)


# ------------------------------------------------------------------
# B. Failure concentration by N
# ------------------------------------------------------------------

allN=df_all.groupby(
    "N"
).size()

badN=w10.groupby(
    "N"
).size()

rate=(
    badN.reindex(
        allN.index,
        fill_value=0
    )
    /
    allN
)

axs[0,1].bar(
    rate.index,
    rate.values,
    width=14,
    color="#0072B2"
)

axs[0,1].set_xlabel(
    "Population size $N$"
)

axs[0,1].set_ylabel(
    "Fraction in worst 10%"
)

axs[0,1].set_title(
    "(B) Failure concentration by $N$"
)


# ------------------------------------------------------------------
# C. R0 versus omega
# ------------------------------------------------------------------

axs[0,2].scatter(
    df_all.R0,
    df_all.omega,
    s=15,
    alpha=.18,
    color=".5",
    label="All test"
)

axs[0,2].scatter(
    w10.R0,
    w10.omega,
    s=32,
    alpha=.85,
    color="#D55E00",
    label="Worst 10%"
)

axs[0,2].axvline(
    1,
    color="black",
    ls="--",
    lw=1
)

axs[0,2].set_xlabel(
    r"$R_0=\beta/\gamma$"
)

axs[0,2].set_ylabel(
    r"$\omega$"
)

axs[0,2].set_title(
    r"(C) Failure regime in $(R_0,\omega)$"
)

axs[0,2].legend(
    frameon=False
)


# ------------------------------------------------------------------
# D. R0 versus E_rho
# ------------------------------------------------------------------

axs[1,0].scatter(
    df_all.R0,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,0].scatter(
    w10.R0,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,0].axvline(
    1,
    color="black",
    ls="--",
    lw=1
)

axs[1,0].set_xlabel(
    r"$R_0=\beta/\gamma$"
)

axs[1,0].set_ylabel(
    r"$E_\rho$"
)

axs[1,0].set_title(
    r"(D) Tail error versus $R_0$"
)


# ------------------------------------------------------------------
# E. Exact overflow versus E_rho
# ------------------------------------------------------------------

axs[1,1].scatter(
    df_all.exact_overflow,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,1].scatter(
    w10.exact_overflow,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,1].set_xlabel(
    r"Exact $P(C>N)$"
)

axs[1,1].set_ylabel(
    r"$E_\rho$"
)

axs[1,1].set_title(
    "(E) Overflow risk"
)


# ------------------------------------------------------------------
# F. Bimodality versus E_rho
# ------------------------------------------------------------------

axs[1,2].scatter(
    df_all.bimodality,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,2].scatter(
    w10.bimodality,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,2].set_xlabel(
    "Bimodality score"
)

axs[1,2].set_ylabel(
    r"$E_\rho$"
)

axs[1,2].set_title(
    "(F) Distributional shape"
)


plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_failure_diagnostics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 5. SIX WORST CASES
# =====================================================================================

worst6=sorted(
    ALL,
    key=lambda r:r[FAIL_METRIC],
    reverse=True
)[:6]


print("\n"+"="*115)
print("SIX WORST TEST CONFIGURATIONS")
print("="*115)


for j,r in enumerate(
    worst6,1
):

    print(
        f"{j}. "
        f"{r['split']:13s} | "
        f"N={r['N']:3d}, "
        f"i0={r['i0']:3d} | "
        f"beta={r['beta']:.3f}, "
        f"gamma={r['gamma']:.3f}, "
        f"omega={r['omega']:.3f} | "
        f"R0={r['R0']:.3f} | "
        f"E2={r['E2']:.4f}, "
        f"E_rho={r['E_rho']:.4f}, "
        f"overflow={r['exact_overflow']:.4f}"
    )


# =====================================================================================
# 6. WORST-6 PMFs — NOW WITH R0
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


for ax,r in zip(
    axs.flat,
    worst6
):

    c=np.arange(
        r["N"]+1
    )

    ax.bar(
        c,
        r["exact_p"][:-1],
        color=".82",
        width=.85,
        label="Exact Markovian"
    )

    ax.plot(
        c,
        r["pred_p"][:-1],
        color="#D55E00",
        lw=1.8,
        label="Neural emulator"
    )

    ax.set_title(
        f"{r['split']}, "
        f"$N={r['N']}$, "
        f"$i_0={r['i0']}$\n"
        rf"$R_0={r['R0']:.2f}$, "
        rf"$E_2={r['E2']:.3f}$, "
        rf"$E_\rho={r['E_rho']:.3f}$"
    )

    ax.set_xlabel(
        "Infection count $c$"
    )

    ax.set_ylabel(
        "Probability mass"
    )


axs[0,0].legend(
    frameon=False
)

plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_worst6_pmfs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 7. WORST-6 TAILS — NOW WITH R0
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


for ax,r in zip(
    axs.flat,
    worst6
):

    c=np.arange(
        r["N"]+1
    )

    ax.plot(
        c,
        r["exact_tail"],
        color="black",
        lw=2,
        label="Exact Markovian"
    )

    ax.plot(
        c,
        r["pred_tail"],
        "--",
        color="#0072B2",
        lw=1.8,
        label="Neural emulator"
    )

    ax.set_ylim(
        -.01,1.01
    )

    ax.set_title(
        f"{r['split']}, "
        f"$N={r['N']}$, "
        f"$i_0={r['i0']}$\n"
        rf"$R_0={r['R0']:.2f}$, "
        rf"$E_\rho={r['E_rho']:.3f}$"
    )

    ax.set_xlabel(
        "Threshold $c$"
    )

    ax.set_ylabel(
        r"$P(C>c)$"
    )


axs[0,0].legend(
    frameon=False
)

plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_worst6_tails.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print("\nFailure analysis saved to:")
print(RESULT_DIR.resolve())

In [ ]:
# =====================================================================================
# FAILURE ANALYSIS — WORST 5% AND 10%
#
# Primary failure metric: E_rho
#
# Includes R0 = beta/gamma:
#   - in tables
#   - in regime diagnostics
#   - in worst-case PMF titles
#   - in worst-case tail titles
# =====================================================================================

from pathlib import Path
import pickle, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


RESULT_DIR=Path(
    "results_section_5_1A_R5000_tailaware_R0"
)

RESULT_FILE=(
    RESULT_DIR
    /
    "section_5_1A_results.pkl"
)

FAIL_METRIC="E_rho"
FRACTIONS=(.05,.10)


with open(
    RESULT_FILE,
    "rb"
) as f:

    D=pickle.load(f)


ALL=D["all_test"]
INTERP=D["interpolation"]


# =====================================================================================
# 1. DATA FRAMES
# =====================================================================================

def to_df(results):

    keep=[
        "index",
        "split",

        "N",
        "i0",
        "i0_fraction",

        "beta",
        "gamma",
        "omega",
        "R0",

        "E2",
        "E_rho",
        "E_overflow",
        "KL",

        "mean_tau_relative_error",
        "var_tau_relative_error",

        "exact_overflow",
        "entropy",
        "bimodality",
        "tau_valid"
    ]

    return pd.DataFrame([
        {
            k:r.get(k,np.nan)
            for k in keep
        }
        for r in results
    ])


df_all=to_df(ALL)
df_interp=to_df(INTERP)


# =====================================================================================
# 2. WORST FRACTIONS
# =====================================================================================

def worst_fraction(
    df,
    fraction,
    metric=FAIL_METRIC
):

    n=max(
        1,
        int(
            math.ceil(
                fraction*len(df)
            )
        )
    )

    return (
        df
        .sort_values(
            metric,
            ascending=False
        )
        .head(n)
        .copy()
    )


W={}


for scope,df in [
    ("all",df_all),
    ("interpolation",df_interp)
]:

    print("\n"+"="*115)
    print(scope.upper())
    print("="*115)

    for frac in FRACTIONS:

        w=worst_fraction(
            df,
            frac
        )

        W[(scope,frac)]=w

        print(
            f"\nWorst {int(frac*100)}% by {FAIL_METRIC}: "
            f"{len(w)}/{len(df)}"
        )

        print(
            f"E_rho: "
            f"median={w.E_rho.median():.4f}, "
            f"min={w.E_rho.min():.4f}, "
            f"max={w.E_rho.max():.4f}"
        )

        print(
            f"R0: "
            f"median={w.R0.median():.3f}, "
            f"IQR=({w.R0.quantile(.25):.3f},"
            f"{w.R0.quantile(.75):.3f}), "
            f"range=({w.R0.min():.3f},"
            f"{w.R0.max():.3f})"
        )

        print(
            "N counts:",
            w["N"]
            .value_counts()
            .sort_index()
            .to_dict()
        )

        print(
            f"i0=1 fraction: "
            f"{(w.i0==1).mean():.3f}"
        )

        print(
            f"median beta={w.beta.median():.3f}, "
            f"gamma={w.gamma.median():.3f}, "
            f"omega={w.omega.median():.3f}"
        )

        print(
            f"median overflow="
            f"{w.exact_overflow.median():.4f}, "
            f"entropy="
            f"{w.entropy.median():.3f}, "
            f"bimodality="
            f"{w.bimodality.median():.3f}"
        )

        w.to_csv(
            RESULT_DIR
            /
            f"worst_{int(frac*100)}pct_{scope}.csv",
            index=False
        )


# =====================================================================================
# 3. ALL TEST VS WORST 10%
# =====================================================================================

w10=W[("all",.10)]

compare_cols=[
    "N",
    "i0_fraction",

    "beta",
    "gamma",
    "omega",
    "R0",

    "exact_overflow",
    "entropy",
    "bimodality",

    "E2",
    "E_rho",
    "KL"
]


comparison=pd.DataFrame({

    "all_test_median":
        df_all[
            compare_cols
        ].median(),

    "worst10_median":
        w10[
            compare_cols
        ].median(),

    "worst10_Q25":
        w10[
            compare_cols
        ].quantile(.25),

    "worst10_Q75":
        w10[
            compare_cols
        ].quantile(.75)

})


print("\n"+"="*115)
print("ALL TEST VERSUS WORST 10%")
print("="*115)

display(comparison)


comparison.to_csv(
    RESULT_DIR
    /
    "failure_regime_summary.csv"
)


# =====================================================================================
# 4. FAILURE DIAGNOSTICS
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


# ------------------------------------------------------------------
# A. Distribution of E_rho
# ------------------------------------------------------------------

axs[0,0].hist(
    df_all.E_rho,
    bins=35,
    color=".72",
    edgecolor="white"
)

q90=df_all.E_rho.quantile(.90)
q95=df_all.E_rho.quantile(.95)

axs[0,0].axvline(
    q90,
    color="#E69F00",
    ls="--",
    label="90th percentile"
)

axs[0,0].axvline(
    q95,
    color="#D55E00",
    ls="--",
    label="95th percentile"
)

axs[0,0].set_xlabel(
    r"$E_\rho$"
)

axs[0,0].set_ylabel(
    "Number of configurations"
)

axs[0,0].set_title(
    "(A) Tail-error distribution"
)

axs[0,0].legend(
    frameon=False
)


# ------------------------------------------------------------------
# B. Failure concentration by N
# ------------------------------------------------------------------

allN=df_all.groupby(
    "N"
).size()

badN=w10.groupby(
    "N"
).size()

rate=(
    badN.reindex(
        allN.index,
        fill_value=0
    )
    /
    allN
)

axs[0,1].bar(
    rate.index,
    rate.values,
    width=14,
    color="#0072B2"
)

axs[0,1].set_xlabel(
    "Population size $N$"
)

axs[0,1].set_ylabel(
    "Fraction in worst 10%"
)

axs[0,1].set_title(
    "(B) Failure concentration by $N$"
)


# ------------------------------------------------------------------
# C. R0 versus omega
# ------------------------------------------------------------------

axs[0,2].scatter(
    df_all.R0,
    df_all.omega,
    s=15,
    alpha=.18,
    color=".5",
    label="All test"
)

axs[0,2].scatter(
    w10.R0,
    w10.omega,
    s=32,
    alpha=.85,
    color="#D55E00",
    label="Worst 10%"
)

axs[0,2].axvline(
    1,
    color="black",
    ls="--",
    lw=1
)

axs[0,2].set_xlabel(
    r"$R_0=\beta/\gamma$"
)

axs[0,2].set_ylabel(
    r"$\omega$"
)

axs[0,2].set_title(
    r"(C) Failure regime in $(R_0,\omega)$"
)

axs[0,2].legend(
    frameon=False
)


# ------------------------------------------------------------------
# D. R0 versus E_rho
# ------------------------------------------------------------------

axs[1,0].scatter(
    df_all.R0,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,0].scatter(
    w10.R0,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,0].axvline(
    1,
    color="black",
    ls="--",
    lw=1
)

axs[1,0].set_xlabel(
    r"$R_0=\beta/\gamma$"
)

axs[1,0].set_ylabel(
    r"$E_\rho$"
)

axs[1,0].set_title(
    r"(D) Tail error versus $R_0$"
)


# ------------------------------------------------------------------
# E. Exact overflow versus E_rho
# ------------------------------------------------------------------

axs[1,1].scatter(
    df_all.exact_overflow,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,1].scatter(
    w10.exact_overflow,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,1].set_xlabel(
    r"Exact $P(C>N)$"
)

axs[1,1].set_ylabel(
    r"$E_\rho$"
)

axs[1,1].set_title(
    "(E) Overflow risk"
)


# ------------------------------------------------------------------
# F. Bimodality versus E_rho
# ------------------------------------------------------------------

axs[1,2].scatter(
    df_all.bimodality,
    df_all.E_rho,
    s=15,
    alpha=.22,
    color="#0072B2"
)

axs[1,2].scatter(
    w10.bimodality,
    w10.E_rho,
    s=32,
    alpha=.85,
    color="#D55E00"
)

axs[1,2].set_xlabel(
    "Bimodality score"
)

axs[1,2].set_ylabel(
    r"$E_\rho$"
)

axs[1,2].set_title(
    "(F) Distributional shape"
)


plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_failure_diagnostics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 5. SIX WORST CASES
# =====================================================================================

worst6=sorted(
    ALL,
    key=lambda r:r[FAIL_METRIC],
    reverse=True
)[:6]


print("\n"+"="*115)
print("SIX WORST TEST CONFIGURATIONS")
print("="*115)


for j,r in enumerate(
    worst6,1
):

    print(
        f"{j}. "
        f"{r['split']:13s} | "
        f"N={r['N']:3d}, "
        f"i0={r['i0']:3d} | "
        f"beta={r['beta']:.3f}, "
        f"gamma={r['gamma']:.3f}, "
        f"omega={r['omega']:.3f} | "
        f"R0={r['R0']:.3f} | "
        f"E2={r['E2']:.4f}, "
        f"E_rho={r['E_rho']:.4f}, "
        f"overflow={r['exact_overflow']:.4f}"
    )


# =====================================================================================
# 6. WORST-6 PMFs — NOW WITH R0
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


for ax,r in zip(
    axs.flat,
    worst6
):

    c=np.arange(
        r["N"]+1
    )

    ax.bar(
        c,
        r["exact_p"][:-1],
        color=".82",
        width=.85,
        label="Exact Markovian"
    )

    ax.plot(
        c,
        r["pred_p"][:-1],
        color="#D55E00",
        lw=1.8,
        label="Neural emulator"
    )

    ax.set_title(
        f"{r['split']}, "
        f"$N={r['N']}$, "
        f"$i_0={r['i0']}$\n"
        rf"$R_0={r['R0']:.2f}$, "
        rf"$E_2={r['E2']:.3f}$, "
        rf"$E_\rho={r['E_rho']:.3f}$"
    )

    ax.set_xlabel(
        "Infection count $c$"
    )

    ax.set_ylabel(
        "Probability mass"
    )


axs[0,0].legend(
    frameon=False
)

plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_worst6_pmfs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================================
# 7. WORST-6 TAILS — NOW WITH R0
# =====================================================================================

fig,axs=plt.subplots(
    2,3,
    figsize=(17,9)
)


for ax,r in zip(
    axs.flat,
    worst6
):

    c=np.arange(
        r["N"]+1
    )

    ax.plot(
        c,
        r["exact_tail"],
        color="black",
        lw=2,
        label="Exact Markovian"
    )

    ax.plot(
        c,
        r["pred_tail"],
        "--",
        color="#0072B2",
        lw=1.8,
        label="Neural emulator"
    )

    ax.set_ylim(
        -.01,1.01
    )

    ax.set_title(
        f"{r['split']}, "
        f"$N={r['N']}$, "
        f"$i_0={r['i0']}$\n"
        rf"$R_0={r['R0']:.2f}$, "
        rf"$E_\rho={r['E_rho']:.3f}$"
    )

    ax.set_xlabel(
        "Threshold $c$"
    )

    ax.set_ylabel(
        r"$P(C>c)$"
    )


axs[0,0].legend(
    frameon=False
)

plt.tight_layout()

plt.savefig(
    RESULT_DIR
    /
    "figure_worst6_tails.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print("\nFailure analysis saved to:")
print(RESULT_DIR.resolve())